# TextMamba3D — A100 Training Pipeline (v4.6)

**V4.6: AttnRes-inspired Cross-Scale Skip Attention + Text Scale Gate**

| Feature | Description |
|---------|-------------|
| Direction A | CrossScaleSkipAttention supplements decoder skip connections with cross-scale context |
| Direction B | TextScaleGate adaptively mixes raw vs text-fused features per scale |
| A100 40GB | batch_size=4, gradient_checkpointing=true, sw_batch_size=2, num_workers=4 |

Config: `configs/textbrats_v8.yaml`

In [ ]:
# Mount Google Drive (run in Colab web UI if using VS Code plugin)
from google.colab import drive
drive.mount('/content/drive')

# Install packages (cached on Drive)
!nvidia-smi 2>/dev/null || echo "No GPU detected (CPU mode)"
!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache mamba-ssm causal-conv1d transformers nibabel tensorboard pyyaml tqdm


In [ ]:
import os, zipfile, shutil

REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'
DRIVE_CODE_ZIP = os.path.join(DRIVE_BASE, 'TextMamba3D_code.zip')
DRIVE_CODE_DIR = os.path.join(DRIVE_BASE, 'TextMamba3D_code')

# Code source priority: VS Code plugin sync > Drive zip > Drive folder
tm_file = os.path.join(REPO_DIR, 'models/textmamba3d.py')
if os.path.exists(tm_file):
    print(f'Local code available at {REPO_DIR} (VS Code plugin)')
elif os.path.exists(DRIVE_CODE_ZIP):
    print(f'Extracting code from {DRIVE_CODE_ZIP}...')
    os.makedirs(REPO_DIR, exist_ok=True)
    with zipfile.ZipFile(DRIVE_CODE_ZIP, 'r') as zf:
        zf.extractall(REPO_DIR)
    print(f'Extracted to {REPO_DIR}')
elif os.path.exists(DRIVE_CODE_DIR):
    print(f'Copying code from {DRIVE_CODE_DIR}...')
    shutil.copytree(DRIVE_CODE_DIR, REPO_DIR)
    print(f'Copied to {REPO_DIR}')
else:
    raise FileNotFoundError(
        f'Code not found. Please either:\n'
        f'  1. Use VS Code Colab plugin to sync local project\n'
        f'  2. Upload TextMamba3D_code.zip to {DRIVE_BASE} on Google Drive'
    )

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

# Extract BraTS data from Drive
DATA_ZIP = os.path.join(DRIVE_BASE, "TextBraTS_data.zip")
DATA_DIR = os.path.join(REPO_DIR, "data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData")

if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    if os.path.exists(DATA_ZIP):
        print(f"Extracting {DATA_ZIP}...")
        with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
            zf.extractall(os.path.dirname(DATA_DIR))
        if os.path.exists(DATA_DIR):
            print(f"Data extracted. Cases: {len(os.listdir(DATA_DIR))}")
        else:
            print(f"ERROR: Expected path not found after extraction: {DATA_DIR}")
            print("Actual contents:", os.listdir(os.path.dirname(DATA_DIR)))
    else:
        print(f"ERROR: {DATA_ZIP} not found on Drive")
else:
    print(f"Data already exists. Cases: {len(os.listdir(DATA_DIR))}")

# Count samples
if os.path.exists(DATA_DIR):
    cases = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
    print(f"Total BraTS cases: {len(cases)}")


## Code Patches (V4.5 + V4.6)

**V4.5 patches:** SeqCA fusion, ET-enriched dataset
**V4.6 patches:** RMSNorm, CrossScaleSkipAttention, TextScaleGate, MultiScaleTextGate + modified decoder + model

All patches are idempotent (safe to re-run).


In [ ]:
import pathlib

# [v4.4-1] Append Sequential Cross-Attention classes to models/fusion.py
fusion_path = pathlib.Path('models/fusion.py')
content = fusion_path.read_text(encoding='utf-8')

if 'SequentialCrossAttention' in content:
    print("[v4.4-1] SeqCA already exists in fusion.py, skipping")
else:
    seqca_code = '''

# Sequential Cross-Attention (TextBraTS-inspired, MICCAI 2025)
# ---------------------------------------------------------------------------

class SequentialCrossAttention(nn.Module):
    """TextBraTS-style Sequential Cross-Attention for text-guided segmentation.

    Two-step cross-attention that reverses the Q/KV direction:
      Step 1 (T2I): Text=Q, Image=KV -> refined features (text-length)
      Step 2 (I2T): Image=Q, Refined=KV -> joint features (image-length)

    This ensures text actively "asks" the image where its descriptions are
    relevant, rather than the image passively querying text tokens.
    """

    def __init__(self, feat_dim: int, text_dim: int, num_heads: int = 4):
        super().__init__()
        assert feat_dim % num_heads == 0, \
            f"feat_dim ({feat_dim}) must be divisible by num_heads ({num_heads})"
        self.num_heads = num_heads
        self.head_dim = feat_dim // num_heads
        self.scale = self.head_dim ** -0.5

        # Project text to image feature dimension
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, feat_dim),
            nn.LayerNorm(feat_dim),
        )

        # Step 1: Text queries Image (T2I)
        self.t2i_norm_q = nn.LayerNorm(feat_dim)
        self.t2i_norm_kv = nn.LayerNorm(feat_dim)
        self.t2i_q = nn.Linear(feat_dim, feat_dim)
        self.t2i_k = nn.Linear(feat_dim, feat_dim)
        self.t2i_v = nn.Linear(feat_dim, feat_dim)
        self.t2i_out = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.LayerNorm(feat_dim),
        )

        # Step 2: Image queries Refined (I2T)
        self.i2t_norm_q = nn.LayerNorm(feat_dim)
        self.i2t_norm_kv = nn.LayerNorm(feat_dim)
        self.i2t_q = nn.Linear(feat_dim, feat_dim)
        self.i2t_k = nn.Linear(feat_dim, feat_dim)
        self.i2t_v = nn.Linear(feat_dim, feat_dim)
        self.i2t_out = nn.Linear(feat_dim, feat_dim)

        # Zero-init Step 2 output for identity-preserving start
        nn.init.zeros_(self.i2t_out.weight)
        nn.init.zeros_(self.i2t_out.bias)

    def _multi_head_attn(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """Multi-head attention computation.

        Args:
            q: [B, Nq, D] queries
            k: [B, Nk, D] keys
            v: [B, Nk, D] values
            mask: [B, Nk] optional mask (1=valid, 0=pad), applied on K dimension
        Returns:
            [B, Nq, D] attention output
        """
        B, Nq, D = q.shape
        H, hd = self.num_heads, self.head_dim

        q = q.reshape(B, Nq, H, hd).transpose(1, 2)   # [B, H, Nq, hd]
        k = k.reshape(B, -1, H, hd).transpose(1, 2)    # [B, H, Nk, hd]
        v = v.reshape(B, -1, H, hd).transpose(1, 2)    # [B, H, Nk, hd]

        attn = (q @ k.transpose(-2, -1)) * self.scale   # [B, H, Nq, Nk]

        if mask is not None:
            attn = attn.masked_fill(
                mask.unsqueeze(1).unsqueeze(2) == 0,     # [B, 1, 1, Nk]
                float('-inf'),
            )

        attn = attn.softmax(dim=-1)
        attn = torch.nan_to_num(attn)  # guard against all-masked rows

        out = (attn @ v).transpose(1, 2).reshape(B, Nq, D)
        return out

    def forward(
        self,
        x: torch.Tensor,
        text_feat: torch.Tensor,
        text_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """
        Args:
            x: [B, N, D] spatial feature tokens (image)
            text_feat: [B, M, D_text] text token features
            text_mask: [B, M] optional mask (1=valid, 0=pad)
        Returns:
            [B, N, D] text-guided features (same shape as input)
        """
        residual = x

        # Project text to image feature space
        text_proj = self.text_proj(text_feat)  # [B, M, D]

        # Step 1: Text=Q, Image=KV
        # Text asks: "where in the image are my descriptions relevant?"
        q1 = self.t2i_q(self.t2i_norm_q(text_proj))    # [B, M, D]
        k1 = self.t2i_k(self.t2i_norm_kv(x))           # [B, N, D]
        v1 = self.t2i_v(self.t2i_norm_kv(x))            # [B, N, D]
        refined = self.t2i_out(self._multi_head_attn(q1, k1, v1))
        # refined: [B, M, D] — text-length, contains image info selected by text

        # Step 2: Image=Q, Refined=KV
        # Image enhances itself using text-selected visual information
        q2 = self.i2t_q(self.i2t_norm_q(x))             # [B, N, D]
        k2 = self.i2t_k(self.i2t_norm_kv(refined))      # [B, M, D]
        v2 = self.i2t_v(self.i2t_norm_kv(refined))       # [B, M, D]
        joint = self.i2t_out(self._multi_head_attn(q2, k2, v2, text_mask))
        # joint: [B, N, D] — image-length

        return residual + joint


class MultiScaleSeqCA(nn.Module):
    """Apply Sequential Cross-Attention at multiple encoder scales."""

    def __init__(self, stage_dims: list[int], text_dim: int, num_heads: int = 4):
        super().__init__()
        self.attn_layers = nn.ModuleList([
            SequentialCrossAttention(dim, text_dim, num_heads=num_heads)
            for dim in stage_dims
        ])

    def forward(
        self,
        features: list[torch.Tensor],
        text_feat: torch.Tensor,
        text_mask: torch.Tensor | None = None,
    ) -> list[torch.Tensor]:
        return [
            attn(feat, text_feat, text_mask)
            for attn, feat in zip(self.attn_layers, features)
        ]


# ---------------------------------------------------------------------------
# MambaFusion: Deep bottleneck fusion via causal Mamba
# ---------------------------------------------------------------------------

class MambaFusion(nn.Module):
    """Fuse image and text features using Mamba.

    Strategy: Concatenate [text, image] tokens, process with Mamba,
    then extract image portion. Text at the front guides image features
    through Mamba's causal nature.
    """

    def __init__(
        self,
        img_dim: int,
        text_dim: int,
        hidden_dim: int,
        depth: int = 2,
        d_state: int = 16,
        dropout: float = 0.0,
    ):
        super().__init__()

        # Project to common dimension
        self.img_proj = nn.Sequential(nn.Linear(img_dim, hidden_dim), nn.LayerNorm(hidden_dim))
        self.text_proj = nn.Sequential(nn.Linear(text_dim, hidden_dim), nn.LayerNorm(hidden_dim))

        # Mamba fusion layers
        self.mamba_fusion = MambaLayer(
            dim=hidden_dim,
            depth=depth,
            d_state=d_state,
            dropout=dropout,
        )

        # Project back to image dimension
        self.out_proj = nn.Linear(hidden_dim, img_dim)
        self.norm = nn.LayerNorm(img_dim)

    def forward(
        self,
        img_feat: torch.Tensor,
        text_feat: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            img_feat: [B, N, D_img] image features
            text_feat: [B, M, D_text] text features
        Returns:
            [B, N, D_img] fused image features
        """
        B, N, _ = img_feat.shape
        M = text_feat.shape[1]

        # Project to common space
        img_h = self.img_proj(img_feat)    # [B, N, hidden_dim]
        text_h = self.text_proj(text_feat)  # [B, M, hidden_dim]

        # Concatenate: [text, image] - text guides image
        concat = torch.cat([text_h, img_h], dim=1)  # [B, M+N, hidden_dim]

        # Mamba fusion
        fused = self.mamba_fusion(concat)  # [B, M+N, hidden_dim]

        # Extract image portion
        img_fused = fused[:, M:, :]  # [B, N, hidden_dim]

        # Project back and residual
        out = self.out_proj(img_fused)
        out = self.norm(out + img_feat)

        return out


# ---------------------------------------------------------------------------
'''
    fusion_path.write_text(content + seqca_code, encoding='utf-8')
    print("[v4.4-1] Appended SeqCA + MultiScaleSeqCA to fusion.py")

# Verify
content = fusion_path.read_text(encoding='utf-8')
assert 'SequentialCrossAttention' in content, "SeqCA not found!"
assert 'MultiScaleSeqCA' in content, "MultiScaleSeqCA not found!"
print(f"  fusion.py size: {len(content)} chars")


In [ ]:
import pathlib

# [v4.6-1] Append V4.6 modules to fusion.py:
#   RMSNorm, CrossScaleSkipAttention, TextScaleGate, MultiScaleTextGate
fusion_path = pathlib.Path('models/fusion.py')
content = fusion_path.read_text(encoding='utf-8')

if 'CrossScaleSkipAttention' in content:
    print("[v4.6-1] V4.6 modules already exist in fusion.py, skipping")
else:
    v46_code = '''

# ---------------------------------------------------------------------------
# Cross-Scale Skip Attention (AttnRes-inspired, V4.6)
# ---------------------------------------------------------------------------

# Custom RMSNorm for compatibility with PyTorch < 2.4 (which lacks nn.RMSNorm)
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization (AttnRes best practice for keys)."""

    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.scale


class CrossScaleSkipAttention(nn.Module):
    """AttnRes-inspired cross-scale skip attention for U-Net decoders.

    SUPPLEMENTS (not replaces) the original matched-level skip connection.
    Uses a learned pseudo-query to attend over global-average-pooled representations
    of ALL available encoder features, producing a cross-scale context vector
    that is broadcast to all spatial positions.

    The original skip provides spatially-detailed per-voxel information.
    This module adds coarse multi-resolution context on top.

    AttnRes best practices: zero-init pseudo-query, RMSNorm on keys, zero-init out_proj.
    """

    def __init__(self, target_dim: int, candidate_dims: list):
        """
        Args:
            target_dim: Channel dim of decoder target (after upsample)
            candidate_dims: Channel dims of each candidate encoder feature
        """
        super().__init__()
        if len(candidate_dims) == 0:
            raise ValueError("candidate_dims must be non-empty")
        self.target_dim = target_dim
        self.num_candidates = len(candidate_dims)

        self.key_projs = nn.ModuleList([
            nn.Linear(cd, target_dim) for cd in candidate_dims
        ])
        self.key_norms = nn.ModuleList([
            RMSNorm(target_dim) for _ in candidate_dims
        ])
        self.val_projs = nn.ModuleList([
            nn.Linear(cd, target_dim) for cd in candidate_dims
        ])

        # Pseudo-query: one learnable vector per module instance (zero-init)
        self.pseudo_query = nn.Parameter(torch.zeros(1, 1, target_dim))

        # Output projection (zero-init for identity start)
        self.out_proj = nn.Linear(target_dim, target_dim)
        nn.init.zeros_(self.out_proj.weight)
        nn.init.zeros_(self.out_proj.bias)

        self.register_buffer('scale', torch.tensor(target_dim ** -0.5))

    def forward(
        self,
        target: torch.Tensor,
        candidates: list,
    ) -> torch.Tensor:
        """
        Args:
            target: [B, L_target, D_target] decoder features after upsample
            candidates: list of [B, L_i, D_i] encoder features (variable lengths)
        Returns:
            [B, L_target, D_target] cross-scale skip contribution (additive)
        """
        assert len(candidates) == self.num_candidates, \
            f"Expected {self.num_candidates} candidates, got {len(candidates)}"
        B, L, _ = target.shape

        keys = []
        values = []
        for i, cand in enumerate(candidates):
            pooled = cand.mean(dim=1, keepdim=True)  # [B, 1, D_i]
            k = self.key_norms[i](self.key_projs[i](pooled))  # [B, 1, D_target]
            v = self.val_projs[i](pooled)                       # [B, 1, D_target]
            keys.append(k)
            values.append(v)

        keys = torch.cat(keys, dim=1)      # [B, S, D_target]
        values = torch.cat(values, dim=1)   # [B, S, D_target]

        q = self.pseudo_query.expand(B, -1, -1)  # [B, 1, D_target]
        attn = (q * self.scale) @ keys.transpose(-2, -1)  # [B, 1, S]
        attn = attn.softmax(dim=-1)

        agg = (attn @ values)  # [B, 1, D_target]
        out = self.out_proj(agg).expand(-1, L, -1).contiguous()  # [B, L, D_target]

        return out


# ---------------------------------------------------------------------------
# Text Scale Gate (AttnRes-inspired adaptive text fusion, V4.6)
# ---------------------------------------------------------------------------

class TextScaleGate(nn.Module):
    """Learned gate controlling text fusion contribution at each scale.

    Adaptively mixes raw encoder features with text-fused features:
        output = gate * fused + (1 - gate) * raw

    Init: zero weights + bias=2.0 → sigmoid(2)≈0.88 → text fusion ON by default.
    """

    def __init__(self, feat_dim: int, init_bias: float = 2.0):
        super().__init__()
        self.gate_proj = nn.Linear(2 * feat_dim, 1)
        nn.init.zeros_(self.gate_proj.weight)
        nn.init.constant_(self.gate_proj.bias, init_bias)

    def forward(self, raw: torch.Tensor, fused: torch.Tensor) -> torch.Tensor:
        """
        Args:
            raw: [B, L, D] raw encoder features (before text fusion)
            fused: [B, L, D] text-fused features (after SeqCA)
        Returns:
            [B, L, D] gated combination
        """
        gate_input = torch.cat([raw, fused], dim=-1)  # [B, L, 2D]
        gate = torch.sigmoid(self.gate_proj(gate_input))  # [B, L, 1]
        return gate * fused + (1 - gate) * raw


class MultiScaleTextGate(nn.Module):
    """Apply TextScaleGate at multiple encoder scales."""

    def __init__(self, stage_dims: list[int], init_bias: float = 2.0):
        super().__init__()
        self.gates = nn.ModuleList([
            TextScaleGate(feat_dim=dim, init_bias=init_bias)
            for dim in stage_dims
        ])

    def forward(
        self,
        raw_features: list[torch.Tensor],
        fused_features: list[torch.Tensor],
    ) -> list[torch.Tensor]:
        assert len(raw_features) == len(self.gates) == len(fused_features), \
            f"Expected {len(self.gates)} features, got raw={len(raw_features)}, fused={len(fused_features)}"
        return [
            gate(raw, fused)
            for gate, raw, fused in zip(self.gates, raw_features, fused_features)
        ]
'''
    fusion_path.write_text(content + v46_code, encoding='utf-8')
    print("[v4.6-1] Appended V4.6 modules to fusion.py")

# Verify
content = fusion_path.read_text(encoding='utf-8')
for cls in ['RMSNorm', 'CrossScaleSkipAttention', 'TextScaleGate', 'MultiScaleTextGate']:
    assert cls in content, f"{cls} not found in fusion.py!"
print(f"  fusion.py final size: {len(content)} chars, all V4.6 modules present")


In [ ]:
import pathlib

# [v4.6-2] Overwrite decoder_3d.py with V4.6 version
# Includes CrossScaleSkipAttention integration (supplements, not replaces, skip connections)
decoder_path = pathlib.Path('models/decoder_3d.py')

decoder_content = '''# models/decoder_3d.py
import torch
import torch.nn as nn
from einops import rearrange
from .mamba_block import CrossScanBiMamba3DLayer
from .fusion import CrossScaleSkipAttention


class PatchExpanding3D(nn.Module):
    """Patch expanding for upsampling."""

    def __init__(self, dim: int, out_dim: int, spatial_dims: tuple):
        super().__init__()
        self.dim = dim
        self.spatial_dims = spatial_dims
        self.expand = nn.Linear(dim, 8 * out_dim, bias=False)
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, C = x.shape
        D, H, W = self.spatial_dims
        x = self.expand(x)
        x = rearrange(x, 'b (d h w) (p1 p2 p3 c) -> b (d p1 h p2 w p3) c',
                     d=D, h=H, w=W, p1=2, p2=2, p3=2)
        x = self.norm(x)
        return x


class MambaDecoder3D(nn.Module):
    """3D Mamba Decoder with skip connections.

    V4.6: Optional CrossScaleSkipAttention supplements (not replaces) the
    existing matched-level skip projection with cross-scale context.
    """

    def __init__(
        self,
        img_size: tuple = (96, 96, 96),
        patch_size: tuple = (4, 4, 4),
        out_channels: int = 4,
        embed_dim: int = 96,
        depths: list = [2, 2, 2, 2],
        d_state: int = 16,
        dropout: float = 0.0,
        use_checkpoint: bool = False,
        deep_supervision: bool = False,
        use_cross_scale_skip: bool = False,
    ):
        super().__init__()
        self.num_stages = len(depths)
        self.deep_supervision = deep_supervision
        self.use_cross_scale_skip = use_cross_scale_skip

        d, h, w = img_size[0] // patch_size[0], \
                  img_size[1] // patch_size[1], \
                  img_size[2] // patch_size[2]

        self.stages = nn.ModuleList()
        self.upsamples = nn.ModuleList()
        self.skip_projs = nn.ModuleList()

        # V4.6: cross-scale attention modules (one per skip connection)
        self.cross_scale_attns = nn.ModuleList() if use_cross_scale_skip else None

        skip_count = 0

        for i in range(len(depths) - 1, -1, -1):
            dim = embed_dim * (2 ** i)
            spatial = (d // (2 ** i), h // (2 ** i), w // (2 ** i))

            stage = CrossScanBiMamba3DLayer(
                dim=dim,
                depth=depths[i],
                spatial_dims=spatial,
                d_state=d_state,
                dropout=dropout,
                use_checkpoint=use_checkpoint,
            )
            self.stages.append(stage)

            if i > 0:
                spatial = (d // (2 ** i), h // (2 ** i), w // (2 ** i))
                upsample = PatchExpanding3D(
                    dim=dim,
                    out_dim=dim // 2,
                    spatial_dims=spatial,
                )
                self.upsamples.append(upsample)

                # Original skip projection (ALWAYS present)
                skip_proj = nn.Linear(dim // 2, dim // 2)
                self.skip_projs.append(skip_proj)

                # V4.6: cross-scale attention (supplemental)
                if use_cross_scale_skip:
                    target_dim = dim // 2
                    skip_idx = len(depths) - 2 - skip_count
                    candidate_dims = [embed_dim * (2 ** j) for j in range(skip_idx + 1)]
                    self.cross_scale_attns.append(
                        CrossScaleSkipAttention(
                            target_dim=target_dim,
                            candidate_dims=candidate_dims,
                        )
                    )
                    skip_count += 1

        # Deep supervision
        if deep_supervision:
            self.aux_heads = nn.ModuleList()
            self.aux_spatials = []
            for idx, i in enumerate(range(len(depths) - 1, 0, -1)):
                dim = embed_dim * (2 ** i)
                spatial = (d // (2 ** i), h // (2 ** i), w // (2 ** i))
                self.aux_heads.append(nn.Conv3d(dim, out_channels, 1))
                self.aux_spatials.append(spatial)

        self.final_expand = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * (patch_size[0] ** 3)),
            nn.GELU(),
        )
        self.final_proj = nn.Conv3d(embed_dim, out_channels, 1)

        self.patch_size = patch_size
        self.base_spatial = (d, h, w)

    def forward(self, features: list) -> torch.Tensor:
        """
        Args:
            features: List of encoder features [stage0, stage1, ..., bottleneck]
        Returns:
            [B, out_channels, D, H, W]
        """
        self._aux_outputs = []
        x = features[-1]

        for i, stage in enumerate(self.stages):
            x = stage(x)

            if self.deep_supervision and self.training and i < len(self.aux_heads):
                B_aux, L_aux, C_aux = x.shape
                ds, hs, ws = self.aux_spatials[i]
                aux = rearrange(x, 'b (d h w) c -> b c d h w', d=ds, h=hs, w=ws)
                self._aux_outputs.append(self.aux_heads[i](aux))

            if i < len(self.upsamples):
                x = self.upsamples[i](x)

                # Skip connection
                skip_idx = len(features) - 2 - i
                if skip_idx >= 0:
                    # Original matched-level skip (always)
                    skip = self.skip_projs[i](features[skip_idx])
                    x = x + skip

                    # V4.6: cross-scale supplemental attention
                    if self.use_cross_scale_skip and self.cross_scale_attns is not None:
                        candidates = features[:skip_idx + 1]
                        x = x + self.cross_scale_attns[i](x, candidates)

        # Final expansion
        B, L, C = x.shape
        d, h, w = self.base_spatial
        p = self.patch_size[0]

        x = self.final_expand(x)
        x = rearrange(x, 'b (d h w) (p1 p2 p3 c) -> b c (d p1) (h p2) (w p3)',
                     d=d, h=h, w=w, p1=p, p2=p, p3=p, c=C)
        x = self.final_proj(x)

        return x
'''

decoder_path.write_text(decoder_content, encoding='utf-8')
print("[v4.6-2] Overwritten decoder_3d.py with V4.6 version")
print(f"  Features: CrossScaleSkipAttention (supplemental), PatchExpanding3D, deep supervision")


In [ ]:
import pathlib

# [v4.6-3] Overwrite textmamba3d.py with V4.6 version
# Uses MultiScaleSeqCA (from V4.4 patch) + V4.6 TextScaleGate + use_cross_scale_skip
tm_path = pathlib.Path('models/textmamba3d.py')

tm_content = '''# models/textmamba3d.py
"""Text-guided 3D medical image segmentation with Mamba architecture."""

from typing import Optional

import torch
import torch.nn as nn

from .decoder_3d import MambaDecoder3D
from .encoder_3d import MambaEncoder3D
from .fusion import MultiScaleSeqCA, MultiScaleTextGate
from .text_encoder import TextMambaEncoder


class TextMamba3D(nn.Module):
    """Text-guided 3D medical image segmentation model using Mamba architecture."""

    def __init__(
        self,
        img_size: tuple[int, int, int] = (96, 96, 96),
        in_channels: int = 4,
        out_channels: int = 4,
        embed_dim: int = 96,
        depths: list[int] = [2, 2, 2, 2],
        patch_size: tuple[int, int, int] = (4, 4, 4),
        text_embed_dim: int = 256,
        text_max_len: int = 256,
        text_depth: int = 4,
        d_state: int = 16,
        dropout: float = 0.0,
        use_pretrained_text: bool = True,
        unfreeze_text_layers: int = 0,
        use_checkpoint: bool = False,
        text_model_path: str | None = None,
        deep_supervision: bool = False,
        # V4.6 new parameters
        use_text_gate: bool = False,
        use_cross_scale_skip: bool = False,
        text_gate_init_bias: float = 2.0,
    ) -> None:
        super().__init__()

        self.text_embed_dim = text_embed_dim
        self.text_max_len = text_max_len
        bottleneck_dim = embed_dim * (2 ** (len(depths) - 1))

        self.img_encoder = MambaEncoder3D(
            img_size=img_size,
            in_channels=in_channels,
            embed_dim=embed_dim,
            depths=depths,
            patch_size=patch_size,
            d_state=d_state,
            dropout=dropout,
            use_checkpoint=use_checkpoint,
        )

        self.text_encoder = TextMambaEncoder(
            embed_dim=text_embed_dim,
            max_len=text_max_len,
            depth=text_depth,
            d_state=d_state,
            dropout=dropout,
            use_pretrained=use_pretrained_text,
            unfreeze_last_n=unfreeze_text_layers,
            model_path=text_model_path,
        )

        # Multi-scale cross-attention: stages 1,2,3 (stage 0 excluded)
        stage_dims = [embed_dim * (2 ** i) for i in range(1, len(depths))]
        self.multi_scale_attn = MultiScaleSeqCA(
            stage_dims=stage_dims,
            text_dim=text_embed_dim,
            num_heads=4,
        )

        # V4.6 Direction B: Text Scale Gate
        if use_text_gate:
            self.text_gate = MultiScaleTextGate(
                stage_dims=stage_dims,
                init_bias=text_gate_init_bias,
            )
        else:
            self.text_gate = None

        # V4.6 Direction A: pass use_cross_scale_skip to decoder
        self.decoder = MambaDecoder3D(
            img_size=img_size,
            patch_size=patch_size,
            out_channels=out_channels,
            embed_dim=embed_dim,
            depths=depths,
            d_state=d_state,
            dropout=dropout,
            use_checkpoint=use_checkpoint,
            deep_supervision=deep_supervision,
            use_cross_scale_skip=use_cross_scale_skip,
        )

        self.img_proj = nn.Sequential(
            nn.Linear(bottleneck_dim, text_embed_dim),
            nn.LayerNorm(text_embed_dim),
        )

    def forward(
        self,
        img: torch.Tensor,
        text_ids: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        return_features: bool = False,
        use_text: bool = True,
    ) -> torch.Tensor | tuple[
        torch.Tensor,
        Optional[torch.Tensor],
        Optional[torch.Tensor],
        Optional[torch.Tensor],
    ]:
        """Forward pass for text-guided 3D segmentation."""
        img_features = self.img_encoder(img)

        has_text = use_text and text_ids is not None
        if has_text:
            text_features = self.text_encoder(text_ids, attention_mask)
            fused = self.multi_scale_attn(
                img_features[1:], text_features, attention_mask
            )
            # V4.6 Direction B: gate text contribution per scale
            if self.text_gate is not None:
                fused = self.text_gate(img_features[1:], fused)
            decoder_features = [img_features[0]] + fused
        else:
            decoder_features = img_features

        seg_output = self.decoder(decoder_features)

        if not return_features:
            return seg_output

        if has_text:
            pixel_feat = decoder_features[-1]
            img_global = self.img_proj(pixel_feat.mean(dim=1))
            text_global = self.text_encoder.get_global_feature(text_features)
            return seg_output, img_global, text_global, pixel_feat
        else:
            return seg_output, None, None, None

    def forward_without_text(self, img: torch.Tensor) -> torch.Tensor:
        """Convenience method for inference without text guidance."""
        return self.forward(img, text_ids=None, use_text=False)
'''

tm_path.write_text(tm_content, encoding='utf-8')
print("[v4.6-3] Overwritten textmamba3d.py with V4.6 version")
print("  Imports: MultiScaleSeqCA, MultiScaleTextGate")
print("  New params: use_text_gate, use_cross_scale_skip, text_gate_init_bias")


In [ ]:
import pathlib, shutil

# [v4.5-1] Patch brats_textbrats_dataset.py: add ET-enriched stochastic selection
ds_path = pathlib.Path('data/brats_textbrats_dataset.py')
ds_content = ds_path.read_text(encoding='utf-8')

if 'et_enriched' in ds_content:
    print("[v4.5-1] ET-enriched patch already applied, skipping")
else:
    NL = chr(10)
    # Add params to __init__
    ds_content = ds_content.replace(
        "        seed: int = 42," + NL + "    ):",
        "        seed: int = 42," + NL +
        "        et_enriched: bool = False," + NL +
        "        enriched_prob: float = 0.5," + NL +
        "    ):"
    )
    # Store new attributes
    ds_content = ds_content.replace(
        "        self.use_text_features = use_text_features" + NL,
        "        self.use_text_features = use_text_features" + NL +
        "        self.et_enriched = et_enriched" + NL +
        "        self.enriched_prob = enriched_prob" + NL
    )
    # Add _load_enriched_text method
    method = (NL +
        '    def _load_enriched_text(self, case_dir, case_name):' + NL +
        '        path = os.path.join(case_dir, f"{case_name}_et_enriched.txt")' + NL +
        '        if os.path.exists(path):' + NL +
        "            with open(path, 'r', encoding='utf-8') as f:" + NL +
        '                return f.read().strip()' + NL +
        '        return None' + NL + NL
    )
    ds_content = ds_content.replace(
        "    def _load_text_features(",
        method + "    def _load_text_features("
    )
    # Replace text loading with stochastic selection
    old_text_load = (
        "        # Load expert text (NO information leakage!)" + NL +
        "        text = self._load_text(case_dir, case_name)"
    )
    new_text_load = (
        "        # Load expert text (NO information leakage!)" + NL +
        "        original_text = self._load_text(case_dir, case_name)" + NL +
        NL +
        "        # LaCLIP stochastic selection" + NL +
        "        if self.et_enriched:" + NL +
        "            enriched = self._load_enriched_text(case_dir, case_name)" + NL +
        "            if self.split == 'train':" + NL +
        "                if enriched and np.random.random() < self.enriched_prob:" + NL +
        "                    text = original_text + ' ' + enriched" + NL +
        "                else:" + NL +
        "                    text = original_text" + NL +
        "            else:" + NL +
        "                text = (original_text + ' ' + enriched) if enriched else original_text" + NL +
        "        else:" + NL +
        "            text = original_text"
    )
    ds_content = ds_content.replace(old_text_load, new_text_load)
    ds_path.write_text(ds_content, encoding='utf-8')
    print("[v4.5-1] Patched ET-enriched stochastic selection into dataset")

# Verify
ds_content = ds_path.read_text(encoding='utf-8')
assert 'et_enriched' in ds_content, "et_enriched param not found!"
assert '_load_enriched_text' in ds_content, "enriched text loader not found!"
print(f"  Dataset patched: {len(ds_content)} chars")


In [ ]:
import pathlib, yaml

# [v4.6-4a] Verify configs/textbrats_v8.yaml exists (A100 40GB)
config_path = pathlib.Path('configs/textbrats_v8.yaml')
assert config_path.exists(), f"Config not found: {config_path}. Check code zip."
with open(config_path) as f:
    cfg = yaml.safe_load(f)
print(f"[v4.6-4a] Config verified: {config_path}")
print(f"  batch_size={cfg['data']['batch_size']}, "
      f"grad_ckpt={cfg['training']['gradient_checkpointing']}, "
      f"use_text_gate={cfg['model']['use_text_gate']}, "
      f"use_cross_scale_skip={cfg['model']['use_cross_scale_skip']}")

# [v4.6-4b] Patch train.py: forward V4.6 config fields to model constructor
NL = chr(10)
train_path = pathlib.Path('train.py')
train_content = train_path.read_text(encoding='utf-8')

if 'use_text_gate' in train_content:
    print("[v4.6-4b] train.py already patched, skipping")
else:
    # Insert V4.6 config forwarding after dropout line
    old_line = "        dropout=config['model'].get('dropout', 0.0),"
    new_lines = (
        "        dropout=config['model'].get('dropout', 0.0)," + NL +
        "        use_text_gate=config['model'].get('use_text_gate', False)," + NL +
        "        use_cross_scale_skip=config['model'].get('use_cross_scale_skip', False)," + NL +
        "        text_gate_init_bias=config['model'].get('text_gate_init_bias', 2.0),"
    )
    train_content = train_content.replace(old_line, new_lines)
    train_path.write_text(train_content, encoding='utf-8')
    print("[v4.6-4b] Patched train.py with V4.6 config forwarding")

# [v4.6-4c] Patch evaluate_full.py: forward V4.6 config fields
# NOTE: evaluate_full.py uses 'model_cfg' alias, NOT 'config["model"]'
eval_path = pathlib.Path('evaluate_full.py')
eval_content = eval_path.read_text(encoding='utf-8')

if 'use_text_gate' in eval_content:
    print("[v4.6-4c] evaluate_full.py already patched, skipping")
else:
    old_line = "        dropout=model_cfg.get('dropout', 0.0),"
    new_lines = (
        "        dropout=model_cfg.get('dropout', 0.0)," + NL +
        "        use_text_gate=model_cfg.get('use_text_gate', False)," + NL +
        "        use_cross_scale_skip=model_cfg.get('use_cross_scale_skip', False)," + NL +
        "        text_gate_init_bias=model_cfg.get('text_gate_init_bias', 2.0),"
    )
    eval_content = eval_content.replace(old_line, new_lines)
    eval_path.write_text(eval_content, encoding='utf-8')
    print("[v4.6-4c] Patched evaluate_full.py with V4.6 config forwarding")

# Post-patch assertions
train_check = pathlib.Path('train.py').read_text(encoding='utf-8')
eval_check = pathlib.Path('evaluate_full.py').read_text(encoding='utf-8')
assert 'use_text_gate' in train_check, "FATAL: train.py V4.6 patch failed!"
assert 'use_text_gate' in eval_check, "FATAL: evaluate_full.py V4.6 patch failed!"
print("All config + script patches applied and verified!")

In [ ]:
import pathlib, re

# [v4.6-5] Patch train.py with TextScaleGate logging (P0-2)
train_path = pathlib.Path('train.py')
src = train_path.read_text(encoding='utf-8')

# 1. Add log_gate_values function before def main():
gate_fn = '''
@torch.no_grad()
def log_gate_values(model, writer, epoch):
    """Log TextScaleGate sigmoid values to TensorBoard."""
    text_gate = getattr(model, 'text_gate', None)
    if text_gate is None:
        return
    gate_values = {}
    for i, gate in enumerate(text_gate.gates):
        bias = gate.gate_proj.bias.item()
        gate_val = torch.sigmoid(torch.tensor(bias)).item()
        scale_name = f'scale_{i+1}'
        gate_values[scale_name] = gate_val
        writer.add_scalar(f'Gate/{scale_name}_bias', bias, epoch)
        writer.add_scalar(f'Gate/{scale_name}_sigmoid', gate_val, epoch)
    vals = [f'{k}={v:.4f}' for k, v in gate_values.items()]
    print(f'  Gate values: {", ".join(vals)}')
    return gate_values


'''

if 'log_gate_values' not in src:
    src = src.replace('\ndef main():', gate_fn + 'def main():')
    print('[PATCH] Added log_gate_values function')
else:
    print('[SKIP] log_gate_values already present')

# 2. Add call in training loop after contrastive_weight logging
gate_call = "        # Log TextScaleGate values (V4.6)\n        log_gate_values(model, writer, epoch)\n"
anchor = "        writer.add_scalar('Loss/contrastive_weight', criterion.contrastive_weight, epoch)"

if 'log_gate_values(model' not in src and anchor in src:
    src = src.replace(anchor, anchor + '\n\n' + gate_call)
    print('[PATCH] Added log_gate_values call in training loop')
elif 'log_gate_values(model' in src:
    print('[SKIP] log_gate_values call already present')

train_path.write_text(src, encoding='utf-8')

# Verify syntax
import ast
ast.parse(src)
print('[OK] train.py syntax valid after gate logging patch')

In [ ]:
import sys, importlib
sys.path.insert(0, '.')

# Force reimport after patches (prefix match to avoid clearing transformers.models.*)
for mod_name in list(sys.modules.keys()):
    if mod_name == 'models' or mod_name.startswith('models.'):
        del sys.modules[mod_name]

# Verify V4.6 modules exist
from models.fusion import (
    SequentialCrossAttention, MultiScaleSeqCA,
    RMSNorm, CrossScaleSkipAttention, TextScaleGate, MultiScaleTextGate,
)
print("V4.4 modules: SequentialCrossAttention, MultiScaleSeqCA")
print("V4.6 modules: RMSNorm, CrossScaleSkipAttention, TextScaleGate, MultiScaleTextGate")

# Verify decoder has CrossScaleSkipAttention support
from models.decoder_3d import MambaDecoder3D
import inspect
sig = inspect.signature(MambaDecoder3D.__init__)
assert 'use_cross_scale_skip' in sig.parameters, "decoder missing use_cross_scale_skip param!"
print("Decoder: use_cross_scale_skip parameter present")

# Verify textmamba3d has V4.6 params
from models.textmamba3d import TextMamba3D
sig = inspect.signature(TextMamba3D.__init__)
for param in ['use_text_gate', 'use_cross_scale_skip', 'text_gate_init_bias']:
    assert param in sig.parameters, f"TextMamba3D missing {param}!"
print("TextMamba3D: use_text_gate, use_cross_scale_skip, text_gate_init_bias present")

# Verify config
import yaml
with open('configs/textbrats_v8.yaml') as f:
    cfg = yaml.safe_load(f)
assert cfg['model']['use_cross_scale_skip'] is True
assert cfg['model']['use_text_gate'] is True
assert cfg['data']['batch_size'] == 4, "A100 batch_size should be 4"
assert cfg['training']['gradient_checkpointing'] is True, "A100 should use grad ckpt"
print("Config: V4.6 features enabled, A100 optimizations confirmed")

print()
print("All V4.6 patches verified!")

In [ ]:
import os, sys
os.chdir(REPO_DIR)

DATA_DIR = "./data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"

# Check if already generated
sample_case = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))[0]
sample_enriched = os.path.join(DATA_DIR, sample_case, f"{sample_case}_et_enriched.txt")

if os.path.exists(sample_enriched):
    count = sum(
        1 for d in os.listdir(DATA_DIR)
        if os.path.isdir(os.path.join(DATA_DIR, d))
        and os.path.exists(os.path.join(DATA_DIR, d, f"{d}_et_enriched.txt"))
    )
    print(f"ET-enriched text already generated for {count} cases, skipping")
    for case in sorted(os.listdir(DATA_DIR))[:3]:
        path = os.path.join(DATA_DIR, case, f"{case}_et_enriched.txt")
        if os.path.exists(path):
            with open(path, 'r') as f:
                print(f"  {case}: {f.read().strip()[:120]}...")
else:
    print("Generating ET-enriched text descriptions from T1ce images...")
    sys.path.insert(0, '.')
    from data.et_text_enrichment import process_all_cases
    results = process_all_cases(DATA_DIR)

    no_enhancement = sum(1 for desc in results.values() if "No significant" in desc)
    total = len(results)
    print(f"Total: {total}, No enhancement: {no_enhancement} ({no_enhancement/total*100:.1f}%)")
    for name, desc in list(results.items())[:5]:
        print(f"  {name}: {desc}")


## Training (A100 40GB)

| Parameter | Value |
|-----------|-------|
| batch_size | 4 |
| gradient_checkpointing | true |
| sw_batch_size | 2 |
| num_workers | 4 |
| gradient_accumulation | 1 |

In [ ]:
import os, shutil, glob

DRIVE_CKPT = os.path.join(DRIVE_BASE, "checkpoints")
os.makedirs(DRIVE_CKPT, exist_ok=True)

def sync_checkpoints_to_drive():
    local_ckpt = os.path.join(REPO_DIR, "checkpoints")
    if not os.path.exists(local_ckpt):
        return
    for f in glob.glob(os.path.join(local_ckpt, "*.pth")):
        dst = os.path.join(DRIVE_CKPT, os.path.basename(f))
        shutil.copy2(f, dst)
    print(f"Synced checkpoints to {DRIVE_CKPT}")

# Clean local checkpoints (fresh start for v4.6)
for f in glob.glob(os.path.join(REPO_DIR, "checkpoints/*.pth")):
    os.remove(f)
print("Cleaned local checkpoints for v4.6 fresh start")
print("(Previous best checkpoints preserved on Drive)")


In [ ]:
os.chdir(REPO_DIR)
os.environ["DRIVE_CKPT_DIR"] = DRIVE_CKPT

# V4.6 training: SeqCA + CrossScaleSkipAttention + TextScaleGate + ET-Enriched (A100 40GB)
!python -u train.py \
    --config configs/textbrats_v8.yaml \
    --no-text-ratio 0.15 \
    --grad-accum 1 \
    2>&1 | tee training_v4.6_a100.log | grep --line-buffered -E "(Epoch [0-9]+:|train_loss=|Best |Error|Traceback)"

# Sync and save
sync_checkpoints_to_drive()

best_ckpt = os.path.join(DRIVE_CKPT, "best_v4.6.pth")
local_best = os.path.join(REPO_DIR, "checkpoints/best.pth")
if os.path.exists(local_best):
    shutil.copy2(local_best, best_ckpt)
    print(f"Best checkpoint saved: {best_ckpt}")

## Evaluation


In [ ]:
os.chdir(REPO_DIR)

ckpt = os.path.join(REPO_DIR, "checkpoints/best.pth")
if not os.path.exists(ckpt):
    ckpt = os.path.join(DRIVE_CKPT, "best_v4.6.pth")

if os.path.exists(ckpt):
    print("=" * 60)
    print("Evaluation: With Text (SeqCA + TextScaleGate + CrossScaleSkip)")
    print("=" * 60)
    !python evaluate_full.py \
        --config configs/textbrats_v8.yaml \
        --checkpoint "{ckpt}" \
        --split test \
        --use-text \
        --overlap 0.5

    print()

    print("=" * 60)
    print("Evaluation: Without Text (fusion bypassed)")
    print("=" * 60)
    !python evaluate_full.py \
        --config configs/textbrats_v8.yaml \
        --checkpoint "{ckpt}" \
        --split test \
        --no-text \
        --overlap 0.5

    print()
    print("=" * 60)
    print("Compare: with-text Dice - without-text Dice = text guidance delta")
    print("V4.5 baseline: Mean Dice 83.48%")
    print("V4.6 target: >= 84%")
    print("=" * 60)
else:
    print(f"No checkpoint found at {ckpt}")
    print("Run training first")

In [ ]:
import matplotlib.pyplot as plt

# Placeholder: fill in actual results after training
v45_dice = {'ET': 0.0, 'TC': 0.0, 'WT': 0.0, 'Mean': 83.48}
v46_dice = {'ET': 0.0, 'TC': 0.0, 'WT': 0.0, 'Mean': 0.0}  # Fill after eval

if v46_dice['Mean'] == 0.0:
    print("V4.6 results not yet filled in.")
    print("Update v45_dice and v46_dice dictionaries after evaluation, then re-run this cell.")
else:
    labels = list(v45_dice.keys())
    v45_vals = list(v45_dice.values())
    v46_vals = list(v46_dice.values())

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Bar chart comparison
    x = range(len(labels))
    w = 0.35
    ax1.bar([i - w/2 for i in x], v45_vals, w, label='V4.5', color='steelblue', alpha=0.8)
    ax1.bar([i + w/2 for i in x], v46_vals, w, label='V4.6 (A100)', color='coral', alpha=0.8)
    ax1.set_ylabel('Dice (%)')
    ax1.set_title('V4.5 vs V4.6 Dice Comparison')
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels)
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)

    # Delta chart
    deltas = [v46 - v45 for v45, v46 in zip(v45_vals, v46_vals)]
    colors = ['green' if d >= 0 else 'red' for d in deltas]
    ax2.bar(labels, deltas, color=colors, alpha=0.8)
    ax2.axhline(y=0, color='black', linewidth=0.5)
    ax2.set_ylabel('Delta (%)')
    ax2.set_title('V4.6 - V4.5 Improvement')
    ax2.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('v46_a100_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: v46_a100_comparison.png")

## Resume Training (After Disconnect)


In [ ]:
import os, shutil
os.chdir(REPO_DIR)
os.environ["DRIVE_CKPT_DIR"] = DRIVE_CKPT

resume_ckpt = os.path.join(DRIVE_CKPT, "last.pth")
if os.path.exists(resume_ckpt):
    print(f"Resuming from {resume_ckpt}")
    !python train.py \
        --config configs/textbrats_v8.yaml \
        --resume "{resume_ckpt}" \
        --no-text-ratio 0.15 \
        --grad-accum 1

    sync_checkpoints_to_drive()

    best_ckpt = os.path.join(DRIVE_CKPT, "best_v4.6.pth")
    local_best = os.path.join(REPO_DIR, "checkpoints/best.pth")
    if os.path.exists(local_best):
        shutil.copy2(local_best, best_ckpt)
        print(f"Best checkpoint saved: {best_ckpt}")
else:
    print("No checkpoint to resume from.")
    print(f"Expected: {resume_ckpt}")
    print("Run training first")